# Aplicação e comparação de IA's de previsão

Para a resolução do problema, vamos usar os modelos:

- LightGBM (Gradient Boosting)
    - Por se tratar de dados tabulares grandes
    - Lida muito bem com variáveis numéricas
    - Lida muito bem com não linearidade
    - Não precisa de normalização

- Random Forest
    - Robusto
    - Não sofre com outliers
    - Lida muito bem com variáveis numéricas
    - Menos sensível a erros de dataset

Existe a possibilidade de usar os dois com o Random Forest validando dataset e checando overfitting e o LightGBM como modelo final para determinar o resultado final (Sugestão do ChatGPT). Esse último método faremos para comparação e verificação se houve melhoria de resultados.

Analisaremos as machine learning separadamente. Depois faremos unificando os dois para avaliar resultado.

Importando Biliotecas que serão utilizadas nos código:

In [40]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

Declarando variáveis comuns para os dois códigos:

In [41]:
CAM_PAD = r"C:\Users\leona\OneDrive\Documentos\MBA Eng Dados\Projeto Final de curso\base de dados"

CAM_TREINO = CAM_PAD + r"\camada_treino.csv"

CAM_TESTE = CAM_PAD + r"\camada_teste.csv"

Importação de dataframes base de treinamento e teste.

In [42]:
pre_treino = pd.read_csv(CAM_TREINO, sep=";", encoding="latin-1")

coluna_cnpj_treino = pre_treino["cnpj_basico"].tolist()

treino = pre_treino.drop(columns=["cnpj_basico"])

pre_teste = pd.read_csv(CAM_TESTE, sep=";", encoding="latin-1")

coluna_cnpj = pre_teste["cnpj_basico"].tolist()

teste_lgbm = pre_teste.drop(columns=["situacao_cadastral","cnpj_basico"])

teste_rf = pre_teste.drop(columns=["situacao_cadastral","cnpj_basico"])

**Observação:** Vamos remover do dataframe teste a coluna "situcao_cadastral" para depois comparar os resultados e determinarmos as diferenças do real e o simulado.

In [53]:
n_estimators=500, # "árvores" do modelo, quanto maior mais complexo, mas cuidado com overfitting
learning_rate=0.05, # taxa de aprendizado de cada "árvore", quanto menor mais lento, mas pode melhorar a performance
random_state=42, # para reprodutibilidade
n_jobs=-1 # usar todos os núcleos disponíveis para acelerar o treinamento

## 1. LightGBM (Gradient Boosting)

In [43]:
def lightgbm(df_train: pd.DataFrame, df_predict: pd.DataFrame, coluna_target: str = "situacao_cadastral", n_estimators=500, learning_rate=0.05, random_state=42, n_jobs=-1):
    """
    Args:
        df_train (pd.DataFrame): Dataset de treino com target.
        df_predict (pd.DataFrame): Dataset para previsão.
        coluna_target (str): Nome da coluna de target.

    Retorno:
        pd.DataFrame: DataFrame com coluna 'failed_or_not' (probabilidade).

    Observação: A função assume que o target é binário, onde a classe positiva é representada pelo valor 8 e 2 para classe negativa. Ajuste conforme necessário para outros casos.
    """

    # Cópias para evitar mutação
    df_train = df_train.copy()
    df_predict = df_predict.copy()

    # Target
    y = (df_train[coluna_target] == 8).astype(int)

    # Features
    X = df_train.drop(columns=[coluna_target], errors="ignore")
    X_pred = df_predict.drop(columns=[coluna_target], errors="ignore")

    # Identificar categóricos (se existirem)
    cat_cols = X.select_dtypes(include=["object", "category"]).columns

    for col in cat_cols:
        X[col] = X[col].astype("category")
        if col in X_pred.columns:
            X_pred[col] = X_pred[col].astype("category")

    # Modelo
    model = LGBMClassifier(
        n_estimators=n_estimators, # "árvores" do modelo, quanto maior mais complexo, mas cuidado com overfitting
        learning_rate=learning_rate, # taxa de aprendizado de cada "árvore", quanto menor mais lento, mas pode melhorar a performance
        random_state=random_state, # para reprodutibilidade
        n_jobs=n_jobs # usar todos os núcleos disponíveis para acelerar o treinamento
    )

    model.fit(X, y)

    # Probabilidade de falência
    probs = model.predict_proba(X_pred)[:, 1]

    # Resultado
    result = df_predict.copy()
    result["failed_or_not"] = probs

    return result

In [44]:
lgbm =  lightgbm(
    df_train = treino, 
    df_predict = teste_lgbm, 
    coluna_target = "situacao_cadastral",
    n_estimators=n_estimators, 
    learning_rate=learning_rate, 
    random_state=random_state, 
    n_jobs=n_jobs
    )

[LightGBM] [Info] Number of positive: 2858228, number of negative: 2805514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.226653 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1548
[LightGBM] [Info] Number of data points in the train set: 5663742, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.504654 -> initscore=0.018615
[LightGBM] [Info] Start training from score 0.018615


In [45]:
print(f"Tipo do resultado: {type(lgbm)}\nTamanho antes: {teste_lgbm.shape}\nTamanho do resultado: {lgbm.shape}")

Tipo do resultado: <class 'pandas.core.frame.DataFrame'>
Tamanho antes: (1415935, 11)
Tamanho do resultado: (1415935, 12)


In [46]:
# Adicionando o CNPJ para facilitar a comparação
lgbm["cnpj_basico"] = coluna_cnpj

display("Prova Real:", pre_teste.head(3), "Previsão LightGBM:", lgbm.head(3))

'Prova Real:'

,cnpj_basico,situacao_cadastral,cnae_fiscal_principal,situacao_especial,empresa_mais_5_anos,tempo_vida,natureza_juridica,capital_social,porte_empresa,tempo_exclusao_mei,tempo_op_simples,tempo_excl_op_simples,cod_reg
0,52538148,8,4712100,NaN,False,1.33,2062,74000.0,1,NaN,0.0,1.39,4
1,34534523,2,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4
2,34534523,2,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4


'Previsão LightGBM:'

,cnae_fiscal_principal,situacao_especial,empresa_mais_5_anos,tempo_vida,natureza_juridica,capital_social,porte_empresa,tempo_exclusao_mei,tempo_op_simples,tempo_excl_op_simples,cod_reg,failed_or_not,cnpj_basico
0,4712100,NaN,False,1.33,2062,74000.0,1,NaN,0.0,1.39,4,0.991650,52538148
1,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4,0.860248,34534523
2,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4,0.860248,34534523


## 2. Random Forest

In [49]:
def random_forest(df_train: pd.DataFrame, df_predict: pd.DataFrame, coluna_target: str = "situacao_cadastral", n_estimators=150, random_state=42, n_jobs=-1):
    """
    Args:
        df_train (pd.DataFrame): Dataset de treino com target.
        df_predict (pd.DataFrame): Dataset para previsão.
        coluna_target (str): Nome da coluna de target no DataFrame de treino.

    Returns:
        pd.DataFrame: DataFrame com coluna 'failed_or_not' (probabilidade).
     Retorno:
        pd.DataFrame: DataFrame com coluna 'failed_or_not' (probabilidade).

    Observação: A função assume que o target é binário, onde a classe positiva é representada pelo valor 8 e 2 para classe negativa. Ajuste conforme necessário para outros casos.
    """

    # Cópias
    df_train = df_train.copy()
    df_predict = df_predict.copy()

    # Target
    y = (df_train[coluna_target] == 8).astype(int)

    # Features
    X = df_train.drop(columns=[coluna_target], errors="ignore")
    X_pred = df_predict.drop(columns=[coluna_target], errors="ignore")

    # Random Forest não aceita categoria diretamente → converter
    X = pd.get_dummies(X, drop_first=True)
    X_pred = pd.get_dummies(X_pred, drop_first=True)

    # Alinhar colunas
    X_pred = X_pred.reindex(columns=X.columns, fill_value=0)

    # Modelo
    model = RandomForestClassifier(
        n_estimators=n_estimators, # "árvores" do modelo, quanto maior mais complexo, mas cuidado com overfitting
        max_depth=None, # sem limite de profundidade, mas pode ser ajustado para evitar overfitting
        random_state=random_state, # para reprodutibilidade
        n_jobs=n_jobs # usar todos os núcleos disponíveis para acelerar o treinamento
    )

    model.fit(X, y)

    probs = model.predict_proba(X_pred)[:, 1]

    result = df_predict.copy()
    result["failed_or_not"] = probs

    return result

In [50]:
rf = random_forest(
    df_train=treino, 
    df_predict=teste_rf, 
    coluna_target="situacao_cadastral",
    n_estimators=150, 
    random_state=random_state, 
    n_jobs=n_jobs
    )

In [51]:
# Adicionando o CNPJ para facilitar a comparação
rf["cnpj_basico"] = coluna_cnpj

print(f"Tipo do resultado: {type(rf)}\nTamanho antes: {teste_rf.shape}\nTamanho do resultado: {rf.shape}")

Tipo do resultado: <class 'pandas.core.frame.DataFrame'>
Tamanho antes: (1415935, 11)
Tamanho do resultado: (1415935, 13)


In [52]:
display("Prova Real:", pre_teste.head(3), "Previsão LightGBM:", rf.head(3))

'Prova Real:'

,cnpj_basico,situacao_cadastral,cnae_fiscal_principal,situacao_especial,empresa_mais_5_anos,tempo_vida,natureza_juridica,capital_social,porte_empresa,tempo_exclusao_mei,tempo_op_simples,tempo_excl_op_simples,cod_reg
0,52538148,8,4712100,NaN,False,1.33,2062,74000.0,1,NaN,0.0,1.39,4
1,34534523,2,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4
2,34534523,2,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4


'Previsão LightGBM:'

,cnae_fiscal_principal,situacao_especial,empresa_mais_5_anos,tempo_vida,natureza_juridica,capital_social,porte_empresa,tempo_exclusao_mei,tempo_op_simples,tempo_excl_op_simples,cod_reg,failed_or_not,cnpj_basico
0,4712100,NaN,False,1.33,2062,74000.0,1,NaN,0.0,1.39,4,0.980000,52538148
1,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4,0.826667,34534523
2,4641903,NaN,False,2.08,2062,110000.0,3,NaN,0.0,2.38,4,0.826667,34534523


A execução do Random Forest demorou 13 min e 19 seg, pelo tempo de demora, vemos que será melhor trabalhar com LightGBM que já é próprio para um grande volume de dados e não há necessidade de seguir a sujestão do ChatGPT de unir as duas. Observando as tabelas com o resultado gerado vemos que os resultados são muito próximos

## 3. Analisando os resultados

O resultado que obtivemos é um número de 0 a 1 que nos diz a probabilidade de falência da empresa de acordo com as informações disponibilizadas.

Para facilitar a aferição de resultados vamos criar alguns intervalos de identificação (Baixa/Média/Alta/Muito Alta).

In [67]:
def categoria_chance(coluna):
    lista = []

    for valor in coluna:
        if valor < 0.35:
            lista.append("Baixa")
        elif valor >= 0.35 and valor < 0.65:
            lista.append("Média")
        elif valor >= 0.65 and valor <= 0.84:
            lista.append("Alta")
        elif valor > 0.84 and valor <= 1:
            lista.append("Muito Alta")
        else:
            lista.append("Valor Inválido")
    
    return lista

- Identificando intervalos

In [68]:
col1 = categoria_chance(lgbm['failed_or_not'].tolist())
col2 = categoria_chance(rf['failed_or_not'].tolist())

lgbm["categoria_chance"] = col1

rf["categoria_chance"] = col2

In [72]:
grouped_lgbm = lgbm.groupby("categoria_chance").size().reset_index(name="count")

grouped_lgbm['perc'] = grouped_lgbm['count'] / grouped_lgbm['count'].sum() * 100

grouped_rf = rf.groupby("categoria_chance").size().reset_index(name="count")

grouped_rf['perc'] = grouped_rf['count'] / grouped_rf['count'].sum() * 100

grouped_pre_teste = pre_teste.groupby("situacao_cadastral").size().reset_index(name="count")

grouped_pre_teste['perc'] = grouped_pre_teste['count'] / grouped_pre_teste['count'].sum() * 100

display('LGBM',grouped_lgbm, 'Random Forest', grouped_rf, 'Prova Real', grouped_pre_teste)

'LGBM'

,categoria_chance,count,perc
0,Alta,32299,2.281108
1,Baixa,680552,48.063788
2,Muito Alta,653104,46.125281
3,Média,49980,3.529823


'Random Forest'

,categoria_chance,count,perc
0,Alta,40165,2.836642
1,Baixa,672763,47.513692
2,Muito Alta,648964,45.832895
3,Média,54043,3.816771


'Prova Real'

,situacao_cadastral,count,perc
0,2,698902,49.359752
1,8,717033,50.640248


Se considerarmos que as chances de Média a Muito Alta forem empresas que faliram de fato temos um percentual somado de 52,5% pelo método Random Forest e 51,94% pelo método LightGBM das empresas analisadas que faliram, percentuais muito próximos das empresas que de fato faliram que foi de 50,6%, sendo uma evidência da qualidade do modelo.